In [76]:
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report

# 1. LOAD DATA
with open('../data/balanced_trading_data.json', 'r') as f:
    data = json.load(f)

# 2. FEATURE ENGINEERING (Flattening the nested JSON)
def preprocess_data(json_data):
    rows = []
    for trade in json_data:
        # Extract components and details as a list of "tags"
        # Example: ["comp_6", "det_67", "det_59"]
        features = []
        for item in trade['selectedComps']:
            features.append(f"comp_{item['component']}")
            for d in item['details']:
                features.append(f"det_{d}")
        
        rows.append({
            'target': trade['result'],
            'features': features
        })
    return pd.DataFrame(rows)

df = preprocess_data(data)

# 3. ONE-HOT ENCODING
# This turns our lists of IDs into columns of 0 and 1
mlb = MultiLabelBinarizer()
X = mlb.fit_transform(df['features'])
y = df['target']
feature_names = mlb.classes_

# 4. TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. THE MODEL (Random Forest)
# We use Random Forest because it is excellent at finding "Multipliers" (Interactions)
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

# 6. INFERRING THE MULTIPLIERS (Feature Importance)
# This shows which IDs had the biggest impact on the outcome
importances = model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature_ID': feature_names,
    'Importance_Weight': importances
}).sort_values(by='Importance_Weight', ascending=False)

print("--- TOP INFERRED MULTIPLIERS ---")
print(feature_importance_df.head(10))

# 7. VALIDATION
y_pred = model.predict(X_test)
print("\n--- MODEL ACCURACY REPORT ---")
print(classification_report(y_test, y_pred))

--- TOP INFERRED MULTIPLIERS ---
   Feature_ID  Importance_Weight
3      comp_4           0.097858
9       det_4           0.092985
11      det_6           0.092586
1      comp_2           0.089623
8       det_3           0.089286
10      det_5           0.087431
5      comp_6           0.080526
6       det_1           0.078842
4      comp_5           0.078530
7       det_2           0.075495

--- MODEL ACCURACY REPORT ---
              precision    recall  f1-score   support

        flat       0.47      0.49      0.48       101
        gain       0.46      0.53      0.49        98
        loss       0.09      0.05      0.06        41

    accuracy                           0.43       240
   macro avg       0.34      0.35      0.34       240
weighted avg       0.40      0.43      0.41       240



In [77]:
import shap
import numpy as np

# 1. Create explainer
explainer = shap.TreeExplainer(model)

# 2. Get the index of the class we want to explain (e.g., 'gain')
# model.classes_ might be ['flat', 'gain', 'loss']
class_idx = list(model.classes_).index('gain')

# 3. Calculate SHAP values for one specific trade
trade_to_explain = X_test[1:2]
shap_values = explainer.shap_values(trade_to_explain)

# 4. Handle the Multi-class structure
# In newer SHAP versions, it might return a single 3D array or a list
if isinstance(shap_values, list):
    # If list, pick the array for the 'gain' class
    current_shap_array = shap_values[class_idx]
else:
    # If 3D array, format is [sample, feature, class]
    current_shap_array = shap_values[0, :, class_idx]

# 5. Extract results
prediction = model.predict(trade_to_explain)[0]
print(f"Model Prediction: {prediction}")

# Find which features were 'active' (value == 1) in this trade
active_indices = np.where(trade_to_explain[0] == 1)[0]

print("\n--- Why did the model choose this? ---")
for idx in active_indices:
    # impact is the contribution of this feature to the 'gain' prediction
    impact = current_shap_array[0, idx] if len(current_shap_array.shape) > 1 else current_shap_array[idx]
    feature_name = feature_names[idx]
    
    direction = "PRO-GAIN" if impact > 0 else "ANTI-GAIN"
    print(f"{feature_name:10} | {direction:10} | Strength: {impact:.4f}")

Model Prediction: gain

--- Why did the model choose this? ---
comp_2     | ANTI-GAIN  | Strength: -0.0080
comp_4     | ANTI-GAIN  | Strength: -0.0073
comp_5     | PRO-GAIN   | Strength: 0.0190
det_2      | PRO-GAIN   | Strength: 0.1522
